# Подготовка данных

Вход — `data/raw/github_codereview_python.csv`. Выход — `data/processed/`: очищенный набор `reviews_clean.csv`,
eval-набор `eval.csv` и сравнение `before_after.csv`.

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

OUT = Path('../data/processed')
OUT.mkdir(parents=True, exist_ok=True)

MAX_DIFF_TOKENS = 1000
MAX_COMMENT_CHARS = 1000
CHARS_PER_TOKEN = 3.5

raw = pd.read_csv('../data/raw/github_codereview_python.csv', keep_default_na=False, na_values=[])
raw.shape

(5248, 21)

## Проверки качества

In [2]:
def normalize(text: str) -> str:
    # CRLF → LF, пробелы в конце строк, пробелы по краям
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    return '\n'.join(line.rstrip() for line in text.split('\n')).strip()

MENTION = re.compile(r'(?<![\w.])@(?!(?:pytest|property|staticmethod|classmethod|dataclass|cache|lru_cache|wraps|patch|override|abstractmethod|user)\b)[A-Za-z0-9][A-Za-z0-9-]{1,38}\b')
HOME = re.compile(r'/home/(?!user\b)[A-Za-z0-9_.-]+')
SUGGESTION = re.compile(r'```suggestion.*?```', re.S)
SECURITY_WORDS = re.compile(r'secur|vulnerab|inject|sanitiz|secret|password|credential|api.?key|token|auth|permission|'
                            r'access control|privilege|xss|csrf|ssl|tls|encrypt|leak|unsafe|exploit|private', re.I)

def text_without_suggestion(comment: str) -> str:
    return SUGGESTION.sub('', comment).strip()

def tokens(text: pd.Series) -> pd.Series:
    return (text.str.len() / CHARS_PER_TOKEN).round().astype(int)

has_comment = ~raw.is_negative
comment = raw.reviewer_comment
security = raw.comment_type == 'security'
security_text = comment[security].map(text_without_suggestion)
dup_key = raw.assign(c=comment.map(normalize))[['repo_name', 'pr_number', 'file_path', 'c']]

issues = pd.Series({
    'пропуски: reviewer_username (у кода без замечаний — ожидаемо)': (raw.reviewer_username == '').sum(),
    'пропуски: diff_context': (raw.diff_context.str.strip() == '').sum(),
    'пропуски: after_code': (raw.after_code.str.strip() == '').sum(),
    'дубли (repo, PR, файл, замечание) после нормализации': dup_key.duplicated().sum(),
    f'diff > {MAX_DIFF_TOKENS} токенов': (tokens(raw.diff_context) > MAX_DIFF_TOKENS).sum(),
    f'замечание > {MAX_COMMENT_CHARS} символов': (comment.str.len() > MAX_COMMENT_CHARS).sum(),
    'CRLF в diff или замечании': (raw.diff_context.str.contains('\r') | comment.str.contains('\r')).sum(),
    '@упоминания в замечаниях': comment.str.contains(MENTION).sum(),
    '/home/<ник> в замечаниях': comment.str.contains(HOME).sum(),
    'security: текст без security-лексики': ((security_text != '') & ~security_text.str.contains(SECURITY_WORDS)).sum(),
    'security: нет текста, только suggestion': (security_text == '').sum(),
})
pd.DataFrame({'строк': issues, 'доля': (issues / len(raw)).round(4)})

,строк,доля
пропуски: reviewer_username (у кода без замечаний — ожидаемо),1726,0.3289
пропуски: diff_context,20,0.0038
пропуски: after_code,1,0.0002
"дубли (repo, PR, файл, замечание) после нормализации",31,0.0059
diff > 1000 токенов,206,0.0393
замечание > 1000 символов,21,0.0040
CRLF в diff или замечании,1021,0.1946
@упоминания в замечаниях,133,0.0253
/home/<ник> в замечаниях,2,0.0004
security: текст без security-лексики,12,0.0023


## Очистка

In [3]:
df = raw.copy()
df['review_id'] = df.index.astype(int)  # номер строки в исходном файле
log = [('исходно', len(df))]

for col in ['diff_context', 'reviewer_comment']:
    df[col] = df[col].map(normalize)

df = df.drop_duplicates(['repo_name', 'pr_number', 'file_path', 'reviewer_comment'])
log.append(('дубли', len(df)))

df = df[df.diff_context != '']
log.append(('пустой diff', len(df)))

df['diff_tokens'] = tokens(df.diff_context)
df = df[df.diff_tokens <= MAX_DIFF_TOKENS]
log.append((f'diff > {MAX_DIFF_TOKENS} токенов', len(df)))

df = df[df.reviewer_comment.str.len() <= MAX_COMMENT_CHARS]
log.append((f'замечание > {MAX_COMMENT_CHARS} символов', len(df)))

df['comment_text'] = np.where(df.is_negative, '', df.reviewer_comment)
df['comment_text'] = df.comment_text.str.replace(MENTION, '@user', regex=True).str.replace(HOME, '/home/user', regex=True)

text = df.comment_text.map(text_without_suggestion)
is_security = df.comment_type == 'security'
df['label_unverified'] = is_security & (text == '')
df['label_fixed'] = is_security & (text != '') & ~text.str.contains(SECURITY_WORDS)
df.loc[df.label_fixed, 'comment_type'] = 'suggestion'
print('security переразмечено:', df.label_fixed.sum(), '| без текста:', df.label_unverified.sum())

pd.DataFrame(log, columns=['шаг', 'строк']).assign(удалено=lambda t: (-t['строк'].diff()).fillna(0).astype(int))

security переразмечено: 10 | без текста: 5


,шаг,строк,удалено
0,исходно,5248,0
1,дубли,5217,31
2,пустой diff,5197,20
3,diff > 1000 токенов,5001,196
4,замечание > 1000 символов,4981,20


## Преобразования и лишние поля

In [4]:
SEVERITY = {'bug': 'blocker', 'security': 'blocker', 'performance': 'major', 'refactor': 'minor', 'suggestion': 'minor',
            'style': 'nit', 'nitpick': 'nit', 'question': 'question', 'none': 'none'}
SEVERITY_ORDER = ['none', 'question', 'nit', 'minor', 'major', 'blocker']
TYPES = sorted(SEVERITY)

df['severity'] = df.comment_type.map(SEVERITY)
df['severity_id'] = df.severity.map(SEVERITY_ORDER.index).astype(int)
df['comment_type_id'] = df.comment_type.map(TYPES.index).astype(int)

COLUMNS = ['review_id', 'repo_name', 'file_path', 'diff_context', 'diff_tokens', 'comment_text', 'comment_type',
           'comment_type_id', 'severity', 'severity_id', 'label_fixed', 'label_unverified']
clean = df[COLUMNS].reset_index(drop=True)
print('удалено колонок:', sorted(set(raw.columns) - set(clean.columns)))
print('comment_type_id:', dict(enumerate(TYPES)))
clean.dtypes.to_frame('тип')

удалено колонок: ['after_code', 'after_lines', 'author_username', 'before_code', 'before_lines', 'comment_length', 'comment_line', 'is_negative', 'language', 'pr_number', 'pr_title', 'quality_score', 'repo_language', 'repo_stars', 'reviewer_comment', 'reviewer_username', 'split']
comment_type_id: {0: 'bug', 1: 'nitpick', 2: 'none', 3: 'performance', 4: 'question', 5: 'refactor', 6: 'security', 7: 'style', 8: 'suggestion'}


,тип
review_id,int64
repo_name,str
file_path,str
diff_context,str
diff_tokens,int64
comment_text,str
comment_type,str
comment_type_id,int64
severity,str
severity_id,int64


In [5]:
clean.severity.value_counts().reindex(SEVERITY_ORDER[::-1])

severity
blocker      202
major         70
minor       2180
nit          143
question     661
none        1725
Name: count, dtype: int64

## Сохранение

In [6]:
clean.to_csv(OUT / 'reviews_clean.csv', index=False)

eval_set = (clean[clean.diff_tokens >= 40]
            .groupby('severity', group_keys=False)
            .sample(n=20, random_state=42)
            [['review_id', 'file_path', 'diff_context', 'comment_text', 'comment_type', 'severity']])
eval_set.to_csv(OUT / 'eval.csv', index=False)

print('reviews_clean.csv:', clean.shape, '| eval.csv:', eval_set.shape)
eval_set.severity.value_counts()

reviews_clean.csv: (4981, 12) | eval.csv: (120, 6)


severity
blocker     20
major       20
minor       20
nit         20
none        20
question    20
Name: count, dtype: int64

## Сравнение «до/после»

In [7]:
PII = ['author_username', 'reviewer_username']

def metrics(t: pd.DataFrame, comment_col: str) -> dict:
    diff, text = t.diff_context, t[comment_col]
    # ключ дублей одинаковый для обоих наборов; pr_number у подготовленного берём из исходного по review_id
    pr = t.pr_number if 'pr_number' in t else raw.pr_number.loc[t.review_id].to_numpy()
    key = pd.DataFrame({'repo': t.repo_name, 'pr': pr, 'file': t.file_path, 'c': text.map(normalize)})
    required = [c for c in ['diff_context', 'file_path', 'comment_type'] if c in t]
    return {
        'строк': len(t),
        'столбцов': t.shape[1],
        'пропуски в diff, файле, типе': int((t[required].astype(str).apply(lambda s: s.str.strip()) == '').any(axis=1).sum()),
        'дубли': int(key.duplicated().sum()),
        f'diff > {MAX_DIFF_TOKENS} токенов': int((tokens(diff) > MAX_DIFF_TOKENS).sum()),
        f'замечание > {MAX_COMMENT_CHARS} символов': int((text.str.len() > MAX_COMMENT_CHARS).sum()),
        'CRLF': int((diff.str.contains('\r') | text.str.contains('\r')).sum()),
        '@упоминания': int(text.str.contains(MENTION).sum()),
        '/home/<ник>': int(text.str.contains(HOME).sum()),
        'колонки с ПДн': sum(c in t for c in PII),
        'метка security': int((t.comment_type == 'security').sum()),
        'security без security-лексики': int(((t.comment_type == 'security') & (text.map(text_without_suggestion) != '')
                                              & ~text.map(text_without_suggestion).str.contains(SECURITY_WORDS)).sum()),
    }

before_after = pd.DataFrame({'исходный': metrics(raw, 'reviewer_comment'), 'подготовленный': metrics(clean, 'comment_text')})
before_after.to_csv(OUT / 'before_after.csv')
before_after

,исходный,подготовленный
строк,5248,4981
столбцов,21,12
"пропуски в diff, файле, типе",20,0
дубли,31,0
diff > 1000 токенов,206,0
замечание > 1000 символов,21,0
CRLF,1021,0
@упоминания,133,0
/home/<ник>,2,0
колонки с ПДн,2,0
